In [1]:
# xreal_rehair/evc/speech2text.py

import os
from dotenv import load_dotenv
from deepgram import DeepgramClient
from pydantic import BaseModel, Field, field_validator
from dataclasses import dataclass
from typing import List
import librosa
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import freqz


class SpeechWord(BaseModel):
    word: str
    start: float
    end: float
    confidence: float = 0.0


class SpeechTextResult(BaseModel):
    transcript: str
    words: list[SpeechWord] = Field(default_factory=list)


load_dotenv()

DEEPGRAM_API_KEY = os.getenv("DEEPGRAM_API_KEY")


def speech_to_text_detail(file_path: str, language: str = "ko-KR") -> SpeechTextResult:

    if not DEEPGRAM_API_KEY:
        raise RuntimeError("DEEPGRAM_API_KEY가 설정되어 있지 않습니다.")

    try:
        deepgram = DeepgramClient(api_key=DEEPGRAM_API_KEY)

        with open(file_path, "rb") as f:
            buffer_data = f.read()

        response = deepgram.listen.v1.media.transcribe_file(
            request=buffer_data,
            model="nova-3",
            language=language,
            filler_words=True,
            utterances=True,
            smart_format=True,
        ).model_dump()

        alternative = response["results"]["channels"][0]["alternatives"][0]
        transcript = alternative.get("transcript", "") or ""

        raw_words = alternative.get("words", []) or []
        words = [
            SpeechWord(
                word=item.get("word", ""),
                start=float(item.get("start", 0.0)),
                end=float(item.get("end", 0.0)),
                confidence=float(item.get("confidence", 0.0)),
            )
            for item in raw_words
            if item.get("word")
        ]

        return SpeechTextResult(transcript=transcript, words=words)

    except Exception as e:
        raise RuntimeError(f"STT 처리 실패: {e}") from e
    
#
speech_text_result = speech_to_text_detail("sample.m4a")
print(speech_text_result)

transcript='안녕하세요, 그 발표를 시작하겠습니다.' words=[SpeechWord(word='안녕하세요', start=0.64, end=1.5999999, confidence=0.5151367), SpeechWord(word='그', start=2.0, end=2.6399999, confidence=0.9301758), SpeechWord(word='발표를', start=3.36, end=4.4, confidence=0.9902344), SpeechWord(word='시작하겠습니다', start=4.48, end=5.6, confidence=0.99609375)]


In [ ]:
import json
import sys
from pathlib import Path


if __package__:
    from .sentence_service import analyze_sentence_pronunciation
else:
    package_directory = Path(__file__).resolve().parent
    sys.path.insert(0, str(package_directory.parent))
    from Legendaryvowels.sentence_service import analyze_sentence_pronunciation


def test_sentence_service() -> None:
    current_directory = Path(__file__).resolve().parent
    audio_path = current_directory / "sample.m4a"

    target_text = "안녕하세요, 고 발표를 시작하겠습니다."

    print("문장형 발음 분석을 시작합니다.")
    print(f"음성 파일: {audio_path}")
    print(f"목표 문장: {target_text}")

    result = analyze_sentence_pronunciation(
        audio_path=str(audio_path),
        target_text=target_text,
    )

    print(
        json.dumps(
            result.model_dump(),
            ensure_ascii=False,
            indent=2,
        )
    )


if __name__ == "__main__":
    test_sentence_service()


In [ ]:
import json
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt


if __package__:
    from .lpc_service import analyze_lpc_by_words
    from .schemas import SpeechTextResult, SpeechWord
else:
    package_directory = Path(__file__).resolve().parent
    sys.path.insert(0, str(package_directory.parent))
    from Legendaryvowels.lpc_service import analyze_lpc_by_words
    from Legendaryvowels.schemas import SpeechTextResult, SpeechWord


SAMPLE_STT = SpeechTextResult(
    transcript="안녕하세요, 그 발표를 시작하겠습니다.",
    words=[
        SpeechWord(
            word="안녕하세요",
            start=0.64,
            end=1.5999999,
            confidence=0.5151367,
        ),
        SpeechWord(
            word="그",
            start=2.0,
            end=2.6399999,
            confidence=0.9301758,
        ),
        SpeechWord(
            word="발표를",
            start=3.36,
            end=4.4,
            confidence=0.9902344,
        ),
        SpeechWord(
            word="시작하겠습니다",
            start=4.48,
            end=5.6,
            confidence=0.99609375,
        ),
    ],
)


def draw_lpc_graph(lpc_result, output_path: Path) -> None:
    grouped = {}
    for segment in lpc_result.segments:
        grouped.setdefault(segment.transcript_index, []).append(segment)

    figure, axes = plt.subplots(
        len(grouped),
        1,
        figsize=(11, max(4, len(grouped) * 3)),
        sharex=True,
        constrained_layout=True,
    )
    if len(grouped) == 1:
        axes = [axes]

    for axis, (transcript_index, segments) in zip(axes, grouped.items()):
        for segment in segments:
            if not segment.analysis.valid_signal:
                continue
            frequencies = [point.frequency_hz for point in segment.analysis.points]
            magnitudes = [point.magnitude_db for point in segment.analysis.points]
            axis.plot(
                frequencies,
                magnitudes,
                linewidth=1.4,
                label=f"S{segment.syllable_index}",
            )

        axis.set_title(
            f"Deepgram word[{transcript_index}] LPC curves"
        )
        axis.set_ylabel("Normalized magnitude (dB)")
        axis.grid(alpha=0.25)
        axis.legend(loc="lower right", ncol=3, fontsize=8)

    axes[-1].set_xlabel("Frequency (Hz)")
    figure.savefig(output_path, dpi=160)
    plt.close(figure)


def main() -> None:
    directory = Path(__file__).resolve().parent
    audio_path = directory / "sample.m4a"
    json_path = directory / "sample_lpc_result.json"
    graph_path = directory / "sample_lpc_plot.png"

    lpc_result = analyze_lpc_by_words(
        str(audio_path),
        SAMPLE_STT.words,
    )
    output = {
        "apiVersion": "v1",
        "sourceAudio": audio_path.name,
        "getlpc": 1,
        "transcript": SAMPLE_STT.transcript,
        "words": [
            {
                "transcriptIndex": index,
                "text": word.word,
                "startSec": word.start,
                "endSec": word.end,
                "sttConfidence": word.confidence,
            }
            for index, word in enumerate(SAMPLE_STT.words)
        ],
        "lpc": lpc_result.model_dump(by_alias=True, mode="json"),
    }
    json_path.write_text(
        json.dumps(output, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    draw_lpc_graph(lpc_result, graph_path)

    print(f"LPC segments: {len(lpc_result.segments)}")
    print(f"JSON: {json_path}")
    print(f"Graph: {graph_path}")


if __name__ == "__main__":
    main()
